# Multi-channel CNN + LSTM cho Vietnamese Text Classification

Bài toán mặc định: **sentiment classification tiếng Việt** với 3 nhãn:

- `0`: negative
- `1`: neutral
- `2`: positive

Kiến trúc:

```text
Input sentence
→ tokenize + pad/truncate
→ embedding 200d
→ CNN channel: Conv1D kernel 3, 5, 7 + global max pooling
→ LSTM channel: LSTM hidden 128
→ concat(CNN features, LSTM feature)
→ Dense 200 + sigmoid
→ Dropout 0.2
→ Linear classifier
```

## 1. Cài đặt thư viện

Chạy cell dưới nếu môi trường của bạn chưa có đủ thư viện. Trên Google Colab nên chạy cell này trước.

In [ ]:
!pip -q install torch datasets scikit-learn tqdm numpy pandas


## 2. Import và cấu hình

In [ ]:
from __future__ import annotations

import json
import os
import random
import re
from collections import Counter
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, List, Sequence, Tuple

import numpy as np
import torch
import torch.nn as nn
from datasets import load_dataset
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"

@dataclass
class TrainConfig:
    task: str = "sentiment"          # "sentiment" hoặc "topic"
    max_features: int = 21540       
    max_len: int = 400               
    embedding_dim: int = 200
    num_filters: int = 150
    kernel_sizes: Tuple[int, ...] = (3, 5, 7)
    lstm_hidden: int = 128
    dense_hidden: int = 200
    dropout: float = 0.2
    batch_size: int = 64             
    epochs: int = 14
    lr: float = 0.002
    seed: int = 1337
    output_dir: str = "outputs_sentiment_cnn_lstm"

cfg = TrainConfig()

cfg

## 3. Reproducibility và tokenizer đơn giản cho tiếng Việt

In [ ]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def basic_vi_tokenize(text: str) -> List[str]:
    text = text.lower().strip()
    text = re.sub(r"([^\w\s])", r" \1 ", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text).strip()
    return text.split() if text else []


def build_vocab(texts: Sequence[str], max_features: int, min_freq: int = 1) -> Dict[str, int]:
    counter: Counter[str] = Counter()
    for text in texts:
        counter.update(basic_vi_tokenize(text))

    vocab = {PAD_TOKEN: 0, UNK_TOKEN: 1}
    for token, freq in counter.most_common(max_features - len(vocab)):
        if freq >= min_freq:
            vocab[token] = len(vocab)
    return vocab


def encode_text(text: str, vocab: Dict[str, int], max_len: int) -> List[int]:
    ids = [vocab.get(tok, vocab[UNK_TOKEN]) for tok in basic_vi_tokenize(text)]
    ids = ids[:max_len]
    if len(ids) < max_len:
        ids += [vocab[PAD_TOKEN]] * (max_len - len(ids))
    return ids

seed_everything(cfg.seed)
print(basic_vi_tokenize("Thầy Huy và thầy Tùng dạy rất hay, nhiệt tình!"))

## 4. Tải dataset UIT-VSFC

In [ ]:
ds = load_dataset("uitnlp/vietnamese_students_feedback")
ds

In [ ]:
for i in range(5):
    print(ds["train"][i])

In [ ]:
# sentiment: 0 negative, 1 neutral, 2 positive
# topic: 0 lecturer, 1 training_program, 2 facility, 3 others

def get_split(split: str, task: str):
    texts = list(ds[split]["sentence"])
    labels = [int(x) for x in ds[split][task]]
    return texts, labels

train_texts, train_labels = get_split("train", cfg.task)
val_texts, val_labels = get_split("validation", cfg.task)
test_texts, test_labels = get_split("test", cfg.task)

print("train:", len(train_texts))
print("validation:", len(val_texts))
print("test:", len(test_texts))
print("label distribution train:", Counter(train_labels))

## 5. Dataset, vocab và DataLoader

In [ ]:
class TextClassificationDataset(Dataset):
    def __init__(self, texts: Sequence[str], labels: Sequence[int], vocab: Dict[str, int], max_len: int):
        self.texts = list(texts)
        self.labels = list(labels)
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int):
        x = encode_text(self.texts[idx], self.vocab, self.max_len)
        y = int(self.labels[idx])
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)

vocab = build_vocab(train_texts, max_features=cfg.max_features)
print("vocab_size:", len(vocab))

train_ds = TextClassificationDataset(train_texts, train_labels, vocab, cfg.max_len)
val_ds = TextClassificationDataset(val_texts, val_labels, vocab, cfg.max_len)
test_ds = TextClassificationDataset(test_texts, test_labels, vocab, cfg.max_len)

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=0)

x_batch, y_batch = next(iter(train_loader))
print("x_batch:", x_batch.shape)
print("y_batch:", y_batch.shape)

## 6. Định nghĩa mô hình Multi-channel CNN + LSTM

In [ ]:
class MultiChannelCnnLstm(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        num_classes: int,
        embedding_dim: int = 200,
        num_filters: int = 150,
        kernel_sizes: Tuple[int, ...] = (3, 7, 5),  # sát thứ tự concat Keras gốc hơn
        lstm_hidden: int = 128,
        dense_hidden: int = 200,
        dropout: float = 0.2,
        pad_idx: int = 0,
    ) -> None:
        super().__init__()

        # Keras gốc dùng 2 Embedding layer riêng
        self.embedding_cnn = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=pad_idx,
        )

        self.embedding_lstm = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=pad_idx,
        )

        self.convs = nn.ModuleList([
            nn.Conv1d(
                in_channels=embedding_dim,
                out_channels=num_filters,
                kernel_size=k,
            )
            for k in kernel_sizes
        ])

        self.relu = nn.ReLU()

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=lstm_hidden,
            batch_first=True,
            bidirectional=False,
        )

        concat_dim = len(kernel_sizes) * num_filters + lstm_hidden

        self.fc = nn.Linear(concat_dim, dense_hidden)
        self.fc_activation = nn.Sigmoid()
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(dense_hidden, num_classes)

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        # CNN channel
        emb_cnn = self.embedding_cnn(input_ids)
        cnn_in = emb_cnn.transpose(1, 2)

        pooled_outputs = []
        for conv in self.convs:
            feat = self.relu(conv(cnn_in))
            pooled = torch.max(feat, dim=2).values
            pooled_outputs.append(pooled)

        cnn_features = torch.cat(pooled_outputs, dim=1)

        # LSTM channel
        emb_lstm = self.embedding_lstm(input_ids)
        _, (h_n, _) = self.lstm(emb_lstm)
        lstm_features = h_n[-1]

        # Fusion + classifier
        features = torch.cat([lstm_features, cnn_features], dim=1)

        x = self.fc_activation(self.fc(features))
        x = self.dropout(x)

        # Return logits. Use nn.CrossEntropyLoss during training.
        logits = self.classifier(x)
        return logits

In [ ]:
num_classes = 3 if cfg.task == "sentiment" else 4
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MultiChannelCnnLstm(
    vocab_size=len(vocab),
    num_classes=num_classes,
    embedding_dim=cfg.embedding_dim,
    num_filters=cfg.num_filters,
    kernel_sizes=cfg.kernel_sizes,
    lstm_hidden=cfg.lstm_hidden,
    dense_hidden=cfg.dense_hidden,
    dropout=cfg.dropout,
    pad_idx=vocab[PAD_TOKEN],
).to(device)

with torch.no_grad():
    logits = model(x_batch[:4].to(device))

print(model)
print("device:", device)
print("logits shape:", logits.shape)

## 7. Hàm evaluate và train

In [ ]:
@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, device: torch.device):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss = 0.0
    all_preds = []
    all_labels = []

    for input_ids, labels in loader:
        input_ids = input_ids.to(device)
        labels = labels.to(device)
        logits = model(input_ids)
        loss = criterion(logits, labels)
        total_loss += loss.item() * labels.size(0)
        preds = logits.argmax(dim=1)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

    return {
        "loss": total_loss / max(1, len(all_labels)),
        "accuracy": accuracy_score(all_labels, all_preds),
        "macro_f1": f1_score(all_labels, all_preds, average="macro"),
        "labels": all_labels,
        "preds": all_preds,
    }


def train_model(model: nn.Module, train_loader: DataLoader, val_loader: DataLoader, cfg: TrainConfig, device: torch.device):
    output_dir = Path(cfg.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adamax(model.parameters(), lr=cfg.lr)

    best_macro_f1 = -1.0
    best_path = output_dir / f"best_{cfg.task}_cnn_lstm.pt"
    history = []

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        running_loss = 0.0
        pbar = tqdm(train_loader, desc=f"epoch {epoch}/{cfg.epochs}")

        for input_ids, labels in pbar:
            input_ids = input_ids.to(device)
            labels = labels.to(device)

            optimizer.zero_grad(set_to_none=True)
            logits = model(input_ids)
            loss = criterion(logits, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

            running_loss += loss.item() * labels.size(0)
            pbar.set_postfix(loss=f"{loss.item():.4f}")

        train_loss = running_loss / len(train_loader.dataset)
        val_metrics = evaluate(model, val_loader, device)

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "val_macro_f1": val_metrics["macro_f1"],
        }
        history.append(row)

        print(
            f"epoch={epoch} "
            f"train_loss={train_loss:.4f} "
            f"val_loss={val_metrics['loss']:.4f} "
            f"val_acc={val_metrics['accuracy']:.4f} "
            f"val_macro_f1={val_metrics['macro_f1']:.4f}"
        )

        if val_metrics["macro_f1"] > best_macro_f1:
            best_macro_f1 = float(val_metrics["macro_f1"])
            torch.save(model.state_dict(), best_path)
            print("saved best checkpoint ->", best_path)

    return history, best_path

## 8. Huấn luyện

In [ ]:
history, best_path = train_model(model, train_loader, val_loader, cfg, device)
history[-3:]

## 9. Đánh giá trên test set

In [ ]:
model.load_state_dict(torch.load(best_path, map_location=device))
test_metrics = evaluate(model, test_loader, device)

print("TEST")
print(f"accuracy={test_metrics['accuracy']:.4f}")
print(f"macro_f1={test_metrics['macro_f1']:.4f}")
print()
print(classification_report(test_metrics["labels"], test_metrics["preds"], digits=4))
print("confusion_matrix:")
print(confusion_matrix(test_metrics["labels"], test_metrics["preds"]))

## 10. So sánh với các phương pháp khác

Phần này tái hiện tinh thần thí nghiệm trong paper: không chỉ train mô hình đề xuất, mà còn so sánh với các baseline riêng lẻ và baseline truyền thống.

Các nhóm được so sánh:

- **TF-IDF + Logistic Regression**
- **TF-IDF + Linear SVM**
- **TF-IDF + Multinomial Naive Bayes**
- **CNN-only**: chỉ giữ nhánh CNN multi-filter.
- **LSTM-only**: chỉ giữ nhánh LSTM.
- **Multi-channel CNN+LSTM**: mô hình chính đã train ở trên.

Metric chính nên dùng: `macro_f1`, vì sentiment dataset thường lệch nhãn; `accuracy` vẫn được giữ để dễ đối chiếu.


In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC


def summarize_predictions(method: str, family: str, y_true, y_pred):
    return {
        "method": method,
        "family": family,
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted"),
    }

comparison_rows = []

# Thêm kết quả của mô hình chính đã train ở trên.
comparison_rows.append({
    "method": "Multi-channel CNN+LSTM",
    "family": "Deep learning",
    "accuracy": test_metrics["accuracy"],
    "macro_f1": test_metrics["macro_f1"],
    "weighted_f1": f1_score(test_metrics["labels"], test_metrics["preds"], average="weighted"),
})

comparison_rows


### 10.1. Baseline truyền thống: TF-IDF + ML classifier

Các baseline này thường chạy nhanh, là mốc rất tốt để kiểm tra xem deep model có thực sự đáng dùng hay không. Với dataset nhỏ/vừa, SVM hoặc Logistic Regression đôi khi rất cạnh tranh.


In [ ]:
classical_models = {
    "TF-IDF + Logistic Regression": LogisticRegression(max_iter=1000, n_jobs=None),
    "TF-IDF + Linear SVM": LinearSVC(),
    "TF-IDF + MultinomialNB": MultinomialNB(),
}

for method_name, classifier in classical_models.items():
    pipe = Pipeline([
        ("tfidf", TfidfVectorizer(
            tokenizer=basic_vi_tokenize,
            token_pattern=None,
            lowercase=False,
            ngram_range=(1, 2),
            min_df=2,
            max_features=50000,
        )),
        ("clf", classifier),
    ])
    pipe.fit(train_texts, train_labels)
    preds = pipe.predict(test_texts)
    row = summarize_predictions(method_name, "TF-IDF baseline", test_labels, preds)
    comparison_rows.append(row)
    print(row)


### 10.2. Deep learning ablation: CNN-only và LSTM-only

Paper nhấn mạnh ý tưởng kết hợp ưu điểm của CNN và LSTM. Vì vậy ta thêm hai ablation quan trọng:

- **CNN-only** kiểm tra sức mạnh của local n-gram features.
- **LSTM-only** kiểm tra sức mạnh của sequential/context features.

Để so sánh công bằng hơn, để `COMPARISON_EPOCHS = cfg.epochs`. Nếu chỉ muốn smoke test nhanh, giảm xuống `1`, `2` hoặc `3`.


In [ ]:
class TextCNNOnly(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        num_classes: int,
        embedding_dim: int = 200,
        num_filters: int = 150,
        kernel_sizes: Tuple[int, ...] = (3, 5, 7),
        dense_hidden: int = 200,
        dropout: float = 0.2,
        pad_idx: int = 0,
    ) -> None:
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.convs = nn.ModuleList([
            nn.Conv1d(embedding_dim, num_filters, kernel_size=k)
            for k in kernel_sizes
        ])
        self.relu = nn.ReLU()
        concat_dim = len(kernel_sizes) * num_filters
        self.fc = nn.Linear(concat_dim, dense_hidden)
        self.fc_activation = nn.Sigmoid()
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(dense_hidden, num_classes)

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        emb = self.embedding(input_ids)
        cnn_in = emb.transpose(1, 2)
        pooled_outputs = []
        for conv in self.convs:
            feat = self.relu(conv(cnn_in))
            pooled_outputs.append(torch.max(feat, dim=2).values)
        features = torch.cat(pooled_outputs, dim=1)
        x = self.fc_activation(self.fc(features))
        x = self.dropout(x)
        return self.classifier(x)


class LSTMOnly(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        num_classes: int,
        embedding_dim: int = 200,
        lstm_hidden: int = 128,
        dense_hidden: int = 200,
        dropout: float = 0.2,
        pad_idx: int = 0,
    ) -> None:
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=lstm_hidden,
            batch_first=True,
            bidirectional=False,
        )
        self.fc = nn.Linear(lstm_hidden, dense_hidden)
        self.fc_activation = nn.Sigmoid()
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(dense_hidden, num_classes)

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        emb = self.embedding(input_ids)
        _, (h_n, _) = self.lstm(emb)
        features = h_n[-1]
        x = self.fc_activation(self.fc(features))
        x = self.dropout(x)
        return self.classifier(x)


In [ ]:
# Đặt False nếu chỉ muốn chạy baseline TF-IDF nhanh.
RUN_DEEP_COMPARISON = True
COMPARISON_EPOCHS = cfg.epochs  # đổi thành 1-3 nếu chỉ smoke test


def train_and_evaluate_deep_baseline(method_name: str, model: nn.Module):
    seed_everything(cfg.seed)
    model = model.to(device)

    baseline_cfg = TrainConfig(**asdict(cfg))
    baseline_cfg.epochs = COMPARISON_EPOCHS
    safe_name = method_name.lower().replace(" ", "_").replace("+", "plus").replace("-", "_")
    baseline_cfg.output_dir = str(Path(cfg.output_dir) / "comparison" / safe_name)

    history_baseline, best_path_baseline = train_model(model, train_loader, val_loader, baseline_cfg, device)
    model.load_state_dict(torch.load(best_path_baseline, map_location=device))
    metrics = evaluate(model, test_loader, device)

    row = {
        "method": method_name,
        "family": "Deep learning ablation",
        "accuracy": metrics["accuracy"],
        "macro_f1": metrics["macro_f1"],
        "weighted_f1": f1_score(metrics["labels"], metrics["preds"], average="weighted"),
    }
    return row, history_baseline, metrics


if RUN_DEEP_COMPARISON:
    cnn_model = TextCNNOnly(
        vocab_size=len(vocab),
        num_classes=num_classes,
        embedding_dim=cfg.embedding_dim,
        num_filters=cfg.num_filters,
        kernel_sizes=cfg.kernel_sizes,
        dense_hidden=cfg.dense_hidden,
        dropout=cfg.dropout,
        pad_idx=vocab[PAD_TOKEN],
    )
    cnn_row, cnn_history, cnn_test_metrics = train_and_evaluate_deep_baseline("CNN-only", cnn_model)
    comparison_rows.append(cnn_row)
    print(cnn_row)

    lstm_model = LSTMOnly(
        vocab_size=len(vocab),
        num_classes=num_classes,
        embedding_dim=cfg.embedding_dim,
        lstm_hidden=cfg.lstm_hidden,
        dense_hidden=cfg.dense_hidden,
        dropout=cfg.dropout,
        pad_idx=vocab[PAD_TOKEN],
    )
    lstm_row, lstm_history, lstm_test_metrics = train_and_evaluate_deep_baseline("LSTM-only", lstm_model)
    comparison_rows.append(lstm_row)
    print(lstm_row)
else:
    print("Skipped deep comparison. Set RUN_DEEP_COMPARISON = True to run CNN-only and LSTM-only.")


### 10.3. Bảng tổng hợp kết quả

Bảng dưới đây là kết quả thực nghiệm trên cùng split train/validation/test. Khi viết báo cáo, nên trích bảng này và ghi rõ cấu hình tokenizer, max length, epoch, batch size, random seed.


In [ ]:
comparison_df = pd.DataFrame(comparison_rows)
comparison_df = comparison_df.sort_values("macro_f1", ascending=False).reset_index(drop=True)

# Làm tròn để bảng dễ đọc.
display(comparison_df.style.format({
    "accuracy": "{:.4f}",
    "macro_f1": "{:.4f}",
    "weighted_f1": "{:.4f}",
}))

comparison_out = Path(cfg.output_dir) / "comparison_results.csv"
comparison_out.parent.mkdir(parents=True, exist_ok=True)
comparison_df.to_csv(comparison_out, index=False)
print("saved comparison table ->", comparison_out.resolve())


## 11. Lưu config, vocab và report


In [ ]:
output_dir = Path(cfg.output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

with open(output_dir / "config.json", "w", encoding="utf-8") as f:
    json.dump(asdict(cfg) | {"vocab_size": len(vocab), "num_classes": num_classes}, f, indent=2, ensure_ascii=False)

with open(output_dir / "vocab.json", "w", encoding="utf-8") as f:
    json.dump(vocab, f, ensure_ascii=False)

report = classification_report(test_metrics["labels"], test_metrics["preds"], digits=4)
with open(output_dir / "test_report.txt", "w", encoding="utf-8") as f:
    f.write(f"accuracy={test_metrics['accuracy']:.4f}\n")
    f.write(f"macro_f1={test_metrics['macro_f1']:.4f}\n\n")
    f.write(report)

print("saved to", output_dir.resolve())

## 12. Inference thử với câu mới


In [ ]:
label_names = {
    "sentiment": ["negative", "neutral", "positive"],
    "topic": ["lecturer", "training_program", "facility", "others"],
}

@torch.no_grad()
def predict(texts: List[str]):
    model.eval()
    input_ids = torch.tensor([encode_text(t, vocab, cfg.max_len) for t in texts], dtype=torch.long).to(device)
    logits = model(input_ids)
    probs = torch.softmax(logits, dim=1).cpu().numpy()
    preds = probs.argmax(axis=1)
    results = []
    for text, pred, prob in zip(texts, preds, probs):
        results.append({
            "text": text,
            "pred_id": int(pred),
            "pred_label": label_names[cfg.task][int(pred)],
            "confidence": float(prob[int(pred)]),
        })
    return results

samples = [
    "giảng viên dạy rất dễ hiểu và nhiệt tình .",
    "phòng học nóng và thiết bị quá cũ .",
    "em chưa có ý kiến về môn học này .",
]

predict(samples)